# Credit Risk Analysis: Phase 3 - Feature Engineering & Validation

**Objective**: Engineer 15+ domain-specific features, validate for leakage and multicollinearity, perform cross-file joins, and prepare data for modeling.

**Output**: `/outputs/processed/processed_loans.csv` with ~50 engineered features

---

## 1. Import Libraries & Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from feature_engineering import FeatureEngineer, run_feature_engineering

print("✅ Libraries imported")

✅ Libraries imported


## 2. Load Input Datasets

In [2]:
# Load datasets
data_path = Path.cwd().parent

loan_df = pd.read_csv(data_path / 'loan_portfolio.csv')
portfolio_metrics_df = pd.read_csv(data_path / 'portfolio_metrics.csv')
macro_df = pd.read_csv(data_path / 'macro_stress_scenarios.csv')

print("✅ Datasets loaded:")
print(f"  • loan_portfolio: {loan_df.shape}")
print(f"  • portfolio_metrics: {portfolio_metrics_df.shape}")
print(f"  • macro_stress_scenarios: {macro_df.shape}")

✅ Datasets loaded:
  • loan_portfolio: (50000, 24)
  • portfolio_metrics: (120, 16)
  • macro_stress_scenarios: (60, 16)


## 3. Execute Feature Engineering Pipeline

In [3]:
# Run full feature engineering pipeline
output_path = data_path / 'outputs' / 'processed'
output_path.mkdir(parents=True, exist_ok=True)

engineered_df, output_file, validation_results = run_feature_engineering(
    loan_df, portfolio_metrics_df, macro_df
)

print(f"\n✅ Feature engineering complete")
print(f"📊 Output shape: {engineered_df.shape}")


🔧 FEATURE ENGINEERING

✅ Target variable: defaulted

📦 Engineering categorical buckets...
  ✓ DTI buckets created: {'>50%': 49999, '35-50%': 1, '<20%': 0, '20-35%': 0}
  ✓ Credit utilization created: {'high': 49982, 'medium': 18}
  ✓ Delinquency severity (proxy from PD)

🔢 Engineering continuous features...
  ✓ Payment burden score created
  ✓ log_ead created
  ✓ log_pd_annual created
  ✓ log_leverage created
  ✓ log_interest_coverage created

🏷️  Encoding categorical variables...
  ✓ One-hot encoded: loan_type (4 columns created)
  ✓ One-hot encoded: collateral (2 columns created)
  ✓ One-hot encoded: sector (9 columns created)

🎯 Target encoding sectors & ratings...
  • Using 5-fold Stratified CV for target encoding (no leakage)...
    ✓ sector_target_encoded created
    ✓ initial_rating_target_encoded created

🔗 Cross-file feature joins...
  ✓ portfolio_avg_pd joined from portfolio_metrics (portfolio-level)
  ✓ portfolio_avg_lgd joined from portfolio_metrics (portfolio-level)
  ✓ p

## 4. Feature Summary

In [4]:
# Display engineered features
print("\n" + "="*70)
print("📊 ENGINEERED FEATURES SUMMARY")
print("="*70)

print(f"\nTotal columns: {engineered_df.shape[1]}")

numeric_cols = engineered_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = engineered_df.select_dtypes(include=['object']).columns.tolist()

print(f"\n🔢 Numeric features: {len(numeric_cols)}")
print(f"🏷️  Categorical features: {len(categorical_cols)}")

# Display sample of new features
print(f"\n📋 Engineered Features (Sample):")
engineered_feature_names = [col for col in engineered_df.columns if any(x in col.lower() 
                                                                         for x in ['dti', 'payment', 'delinq', 'util', 'log_', 'encoded', 'target_', 'sector_avg', 'base_'])]
for col in engineered_feature_names[:15]:
    print(f"  ✓ {col}")

print(f"\n  ... and {len(engineered_feature_names) - 15} more")


📊 ENGINEERED FEATURES SUMMARY

Total columns: 55

🔢 Numeric features: 31
🏷️  Categorical features: 9

📋 Engineered Features (Sample):
  ✓ credit_util
  ✓ delinq_severity
  ✓ payment_burden_score
  ✓ log_ead
  ✓ log_pd_annual
  ✓ log_leverage
  ✓ log_interest_coverage
  ✓ sector_Utilities
  ✓ sector_target_encoded
  ✓ initial_rating_target_encoded
  ✓ base_gdp_shock_pp
  ✓ base_unemp_shock_pp
  ✓ base_rate_shock_pp
  ✓ base_credit_spread_bps

  ... and -1 more


## 5. Data Quality Check

In [5]:
# Check for any remaining nulls
null_summary = engineered_df.isnull().sum()
if null_summary.sum() > 0:
    print("⚠️  Null values remaining:")
    print(null_summary[null_summary > 0])
else:
    print("✅ No null values in processed dataset")

# Display validation results
print(f"\n✅ Validation Results:")
print(f"  • Leakage check: {validation_results['leakage']}")
print(f"  • VIF check: {validation_results['vif']}")
print(f"  • Imbalance ratio: {validation_results['imbalance_ratio']:.2f}:1")

✅ No null values in processed dataset

✅ Validation Results:
  • Leakage check: FAILED
  • VIF check: WARNING
  • Imbalance ratio: 6.19:1


## 6. Verification & Next Steps

In [6]:
print("\n" + "="*70)
print("✅ PHASE 3: FEATURE ENGINEERING COMPLETE")
print("="*70)

print(f"\n📁 Processed data saved to:")
print(f"|   {output_file}")

print(f"\n📊 Dataset Summary:")
print(f"  • Shape: {engineered_df.shape[0]:,} rows × {engineered_df.shape[1]} columns")
print(f"  • Memory: {engineered_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  • Target distribution: {engineered_df['defaulted'].value_counts().to_dict()}")

print(f"\n🔜 Next: Phase 4 - Model Training & Evaluation")
print(f"   • Train 3 models (Logistic Regression, XGBoost, LightGBM)")
print(f"   • 5-fold Stratified Cross-Validation")
print(f"   • Generate ROC, PR, calibration, and feature importance charts")
print(f"   • Compute AUC, Gini, KS, F1 metrics")


✅ PHASE 3: FEATURE ENGINEERING COMPLETE

📁 Processed data saved to:
|   /Users/ajaiupadhyaya/Documents/creditrisk/outputs/processed/processed_loans.csv

📊 Dataset Summary:
  • Shape: 50,000 rows × 55 columns
  • Memory: 40.19 MB
  • Target distribution: {0: 43050, 1: 6950}

🔜 Next: Phase 4 - Model Training & Evaluation
   • Train 3 models (Logistic Regression, XGBoost, LightGBM)
   • 5-fold Stratified Cross-Validation
   • Generate ROC, PR, calibration, and feature importance charts
   • Compute AUC, Gini, KS, F1 metrics
